[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week10_alerting_oncall_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 10 -- Alerting and On-Call (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** Alert threshold calibration, runbook with decision tree, incident simulation, on-call rotation

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Calibrate alert thresholds for all four Grafana panels using 3-sigma bounds from 90-day history
- Write a runbook with a diagnostic decision tree covering all four alert types
- Simulate a data drift incident and verify the runbook diagnoses it correctly in under 15 minutes
- Simulate a model degradation incident and verify correct diagnosis
- Design the on-call rotation for Temi, Sola, and Dr. Emeka



---

## MONDAY -- "Alert Threshold Calibration"


3-sigma from the 30-day rolling mean: for a normally distributed metric, a 3-sigma event occurs by chance once every 370 days -- rare enough to take seriously, common enough to catch real incidents quickly.

Pause and Predict -- the approval rate has a daily standard deviation of about 1.8pp. At 3-sigma, what is the alert threshold deviation? Is this tighter or looser than the 5pp deviation in the SLA? Why should the alert threshold be tighter than the SLA?


### Exercise 10.1 -- 3-sigma calibration

Using 90 days of historical approval rate data, calibrate thresholds at 2-sigma, 3-sigma, and 4-sigma. For each: how many false positives would it have generated in 90 days?


In [ ]:
# Exercise 10.1: 3-sigma calibration
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: Alert Threshold Calibration --
import numpy as np, pandas as pd

def calibrate_threshold(metric_history: pd.Series, n_sigma: float = 3.0) -> dict:
    mu, std = metric_history.mean(), metric_history.std()
    return {'mean': round(float(mu), 4), 'std': round(float(std), 4),
            'lower_bound': round(float(mu - n_sigma * std), 4),
            'upper_bound': round(float(mu + n_sigma * std), 4)}

history = pd.read_csv('monitoring/metric_history_90d.csv')
for metric in ['approval_rate', 'default_probability_mean', 'p99_latency_ms']:
    t = calibrate_threshold(history[metric])
    print(f'{metric}: alert if < {t["lower_bound"]} or > {t["upper_bound"]} '
          f'(mean={t["mean"]}, 3-sigma={3*t["std"]:.4f})')


### Expected Output (illustrative — depends on your 90-day metric history)

```
approval_rate: alert if < 0.6842 or > 0.9124 (mean=0.7983, 3-sigma=0.1141)
default_probability_mean: alert if < 0.0912 or > 0.2734 (mean=0.1823, 3-sigma=0.0911)
p99_latency_ms: alert if < 42.0 or > 178.0 (mean=110.0, 3-sigma=68.0)
```
3-sigma bounds are data-driven, not guessed — they come directly from the mean and standard
deviation of your own metric history. This means the thresholds tighten automatically as your
system's behaviour stabilises over time, and loosen if the underlying metric is naturally
noisy.


### Monday Morning Moment

*Slack -- Monday, 10:00am.*

**Temi Adeyemi:** What approval rate deviation should trigger an alert?

**Dr. Emeka Obi:** What is the natural daily variance in the historical data?

**Temi Adeyemi:** About 1.8pp standard deviation.

**Dr. Emeka Obi:** 3-sigma is 5.4pp. Alert at 5pp deviation from the 30-day mean.

**Temi Adeyemi:** The SLA says 5pp.

**Dr. Emeka Obi:** The alert fires before the SLA is breached, not at the moment of breach. Alert at 4pp. Page at 6pp. SLA breach at 10pp. Three levels.

**Sola Fashola:** Never a single threshold.




---

## TUESDAY -- "The Runbook"


A runbook answers: when this alert fires, what do I check first? The decision tree structure prevents the most common on-call mistake: investigating the system you know best rather than the one that actually failed.


### Exercise 10.2 -- Runbook draft

Write the complete runbook covering all four alert types with a decision tree, escalation paths with names, and the rollback command verbatim.


In [ ]:
# Exercise 10.2: Runbook draft
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: The Runbook --
RUNBOOK = """
## CREDITDEFAULT-NG -- INCIDENT RUNBOOK v1.0

## STEP 1: Identify the signal
  Red infrastructure -> STEP 2a
  Red drift panel    -> STEP 2b
  Red model quality  -> STEP 2c
  Red approval rate  -> check drift first (STEP 2b), then STEP 2c
  Multiple red       -> infrastructure first (STEP 2a)

## STEP 2a: Infrastructure
  1. curl http://credit-model.dataflow.ng/health
     200 -> not infrastructure, go to STEP 2b
     503/timeout -> docker restart credit-model-prod
  2. docker logs credit-model-prod --tail 50
  3. Escalate to Sola if not resolved in 20 minutes

## STEP 2b: Data drift
  1. python monitoring/quick_drift_check.py  (< 30 seconds)
  2. Which features have PSI > 0.25?
  3. git log --oneline --since=yesterday etl/
  4. If ETL change: engineering review (not emergency)
  5. If PSI > 0.35 on any feature: airflow dags trigger credit_model_retraining
  6. Escalate to Dr. Emeka if retraining required

## STEP 2c: Model quality
  1. python monitoring/quick_eval.py --n 200  (< 2 min)
  2. AUC < 0.80 WITH drift: go to STEP 2b
  3. AUC < 0.80 WITHOUT drift: feast materialize, then recheck
  4. Still low: python scripts/rollback.py
  5. Escalate to Dr. Emeka immediately after rollback
"""


### Expected Output (illustrative)

There's no console output — the deliverable is the runbook text itself. The design detail worth
noticing: **Step 1 branches on which panel is red**, not on a guess about what's wrong. A
runbook that starts with "check if it's infrastructure" every time, even when the drift panel
is the one flashing red, wastes the first several minutes of every incident.



---

## WEDNESDAY -- "Incident Simulation 1: Data Drift"


Inject 500 rows with extreme LIMIT_BAL values. Run the monitoring pipeline. Verify: drift alert fires, Grafana drift panel turns red, runbook STEP 2b correctly diagnoses and resolves it.


### Exercise 10.3 -- Drift simulation

Inject extreme LIMIT_BAL values. Run monitoring pipeline. Follow the runbook to diagnose. Record time-to-diagnosis.


In [ ]:
# Exercise 10.3: Drift simulation
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: Incident Simulation 1: Data Drift --
import pandas as pd, numpy as np

current = pd.read_csv('datasets/creditdefault_current.csv')
outliers = pd.DataFrame({
    'LIMIT_BAL':   np.random.uniform(2_000_000, 5_000_000, 500),
    'PAY_0':       np.random.choice([-1, 0, 1, 2], 500),
    'PAY_2':       np.random.choice([-1, 0, 1], 500),
    'BILL_AMT1':   np.random.uniform(0, 1_000_000, 500),
    'PAY_AMT1':    np.random.uniform(0, 100_000, 500),
    'DEFAULT_NEXT_MONTH': np.random.choice([0, 1], 500, p=[0.77, 0.23]),
})
corrupted = pd.concat([current, outliers], ignore_index=True)
corrupted.to_csv('datasets/creditdefault_current_corrupted.csv', index=False)

r = run_drift_monitor('datasets/creditdefault_current_corrupted.csv', 'drift_sim')
print(f'Alert fires: {r["drift_share"] > 0.25}')


### Expected Output (exact structure; drift_share value depends on your reference data)

```
Alert fires: True
```
This cell deliberately injects 500 extreme outlier rows into a copy of the current batch, then
re-runs the same `run_drift_monitor` from Monday's Week 9 exercise against it. If `Alert fires`
comes back `False`, your drift threshold or feature set from Week 9 needs revisiting before you
trust it in a real incident.



---

## THURSDAY -- "Incident Simulation 2: Model Degradation"


Replace the production model with one trained on shuffled labels. Verify: model quality panel detects AUC degradation, drift panel stays green, runbook STEP 2c diagnoses it as model failure (not drift) and triggers rollback.

Pause and Predict -- how would you distinguish model degradation from data drift by looking at the dashboard alone? Which combination of panel states indicates each?


### Exercise 10.4 -- On-call rotation

Design the rotation for Temi, Sola, and Dr. Emeka. 7-day shifts. Specify primary, secondary, escalation path, maximum pages before escalation, quiet hours, handover checklist.


In [ ]:
# Exercise 10.4: On-call rotation
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: Incident Simulation 2: Model Degradation --
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
import mlflow.sklearn

train = pd.read_csv('datasets/creditdefault_q1.csv')
pos_idx = train[train.DEFAULT_NEXT_MONTH==1].sample(frac=0.5, random_state=42).index
train.loc[pos_idx, 'DEFAULT_NEXT_MONTH'] = 0  # flip 50% of positives

X, y = train.drop('DEFAULT_NEXT_MONTH', axis=1), train['DEFAULT_NEXT_MONTH']
bad = RandomForestClassifier(n_estimators=100, random_state=42)
bad.fit(X, y)

with mlflow.start_run():
    mlflow.sklearn.log_model(bad, 'credit_default_model')
    mlflow.log_metric('val_auc', 0.57)  # bad
    print('Bad model logged -- watch the model quality panel')


### Expected Output (exact structure — this deliberately trains a degraded model)

```
Bad model logged -- watch the model quality panel
```
Flipping 50% of the positive labels before training is a controlled way to produce a model
with `val_auc` near random (0.57, as logged) without touching production — use this to confirm
your Grafana model-quality panel actually goes red before you rely on it during a real
incident.



---

## FRIDAY -- "The Friday Build: On-Call Drill"


Dr. Emeka triggers one incident without telling Temi which one. Temi follows the runbook from STEP 1. Time limit: 15 minutes to correct diagnosis. After the drill, update any ambiguous runbook steps. The runbook is a living document.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: On-Call Drill --
# On-call drill evaluation:
# PASS: correct diagnosis using the runbook decision tree within 15 minutes
# PASS: resolution action taken after diagnosis confirmed
# FAIL: diagnosis by intuition rather than by the decision tree
# FAIL: resolution attempted before diagnosis confirmed
# FAIL: wrong incident type diagnosed (drift vs model degradation)

# After the drill:
# Which runbook step was ambiguous?
# Update it to be unambiguous before end of day
print('Post-drill: update runbook, commit, push')


### Expected Output (exact — pure `print()`, no external dependency)

```
Post-drill: update runbook, commit, push
```
The pass/fail criteria above are the actual grading rubric for this week's capstone-style
drill — notice that "reached the right answer" isn't sufficient on its own; *how* you reached
it (decision tree vs intuition, diagnosis before action) is graded too, because in a real
incident the process is what generalises to the next, different incident.


### Friday Workplace Moment

*DataFlow -- Friday on-call drill.*

**Dr. Emeka Obi:** Walk me through it.

**Temi Adeyemi:** STEP 1: infrastructure green, drift green, model quality red. Approval rate 28%, down from 34%. STEP 2c: ran quick_eval.py. AUC 0.59. No drift. Runbook says check feature store, then rollback. Feature store fresh. Ran rollback.py.

**Dr. Emeka Obi:** Time?

**Temi Adeyemi:** 11 minutes.

**Dr. Emeka Obi:** PASS. Which runbook step was ambiguous?

**Temi Adeyemi:** STEP 2c says 'check feature store freshness' but does not say how. Updated to: 'run feast materialize --verify'.

**Dr. Emeka Obi:** Update committed before end of day. That is the standard.



---

## 🎁 BONUS EXERCISE — Filing and Triaging a GitHub Issue from an Incident

*Not required for the core path. Recommended for practice with issue-tracker workflows.*

**Scenario:** During Wednesday's drift simulation, the on-call engineer who ran the drill
noticed the runbook's Step 2b was ambiguous about *which* PSI threshold to check first when
multiple features show elevated PSI simultaneously.

**Your task:**
1. File a GitHub Issue using this structure:
   ```
   Title: Runbook Step 2b ambiguous when multiple features drift simultaneously

   ## Summary
   [one sentence]

   ## Steps to reproduce the ambiguity
   [what triggered the confusion during the drill]

   ## Expected runbook behaviour
   [what Step 2b should specify]

   ## Severity
   [P1/P2/P3 — justify the choice]

   ## Suggested fix
   [your proposed rewording of Step 2b]
   ```
2. Label it appropriately (e.g., `runbook`, `on-call`, `P2`).
3. Write the triage decision: does this block the next on-call rotation, or can it wait for
   the next scheduled runbook review? Justify using the severity you assigned.
4. Link this issue to the postmortem from Wednesday's incident simulation — a good issue
   references the evidence that motivated it, not just the proposed fix.

**Why this matters:** the gap between "we noticed a problem in a drill" and "the problem is
fixed" is exactly the gap an issue tracker exists to close. An observation that lives only in
someone's memory of the drill doesn't survive that person's next vacation.


## YOUR WEEK 10 CHECKLIST

Check each item before moving on.

- [ ] Alert thresholds calibrated to 3-sigma with three severity levels: warning, page, SLA breach.
- [ ] Runbook has a decision tree diagnosing both simulated incident types.
- [ ] Both simulations correctly diagnosed using the runbook, not intuition.
- [ ] On-call rotation covers all 7 days with primary, secondary, and manager escalation.
- [ ] On-call drill: correct diagnosis in under 15 minutes.


## Challenge Task

> **Core path:** Implement three-level alert severity: Warning (Slack), Error (page primary), Critical (page primary + secondary).

> **Advanced path:** Build a post-incident scorecard: compute time-to-detect, time-to-diagnose, time-to-resolve for each simulation.


## Common Mistakes

**Alert threshold equals SLA threshold:** Alert when trending toward SLA breach, not when it has already breached.

**Runbook without a decision tree:** 'Investigate the system' is not actionable at 3am.

**On-call without secondary escalation:** If primary is unreachable, who gets paged?



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)